In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, from_unixtime, to_timestamp, to_date,
    length, trim, lower, concat_ws
)
from pyspark.sql.types import *

In [4]:
spark = SparkSession.builder \
    .appName("wsb_data_pipeline") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/19 19:11:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
spark.version

'3.5.6'

In [6]:
RAW_SUBMISSIONS_PATH = "/project/macss/amritap1/redditproject/data/submissions/*.json"
RAW_COMMENTS_PATH = "/project/macss/amritap1/redditproject/data/comments/*.json"

PROCESSED_SUBMISSIONS_PATH = "/project/macss/amritap1/redditproject/data/processed/submissions_parquet"
PROCESSED_COMMENTS_PATH = "/project/macss/amritap1/redditproject/data/processed/comments_parquet"

In [5]:
subs_test = spark.read.json("/project/macss/amritap1/redditproject/data/submissions/RS_2020-01_subreddit.json")
comments_test = spark.read.json("/project/macss/amritap1/redditproject/data/comments/RC_2020-01_subreddit.json")

print("Submissions rows:", subs_test.count())
print("Comments rows:", comments_test.count())

26/04/19 18:06:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Submissions rows: 11185
Comments rows: 512967


#Data Loading Check (Sanity Test)

We successfully loaded a sample of the raw Reddit data (January 2020) for both submissions and comments using PySpark.

**Results:**
- Submissions: ~11K rows  
- Comments: ~513K rows  

This confirms:
- File paths are correct  
- JSON data is being read properly by Spark  
- The dataset structure aligns with expectations (comments >> submissions)

# Raw Data Inspection

In [6]:
subs_test.printSchema()
comments_test.printSchema()

root
 |-- all_awardings: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- award_sub_type: string (nullable = true)
 |    |    |-- award_type: string (nullable = true)
 |    |    |-- coin_price: long (nullable = true)
 |    |    |-- coin_reward: long (nullable = true)
 |    |    |-- count: long (nullable = true)
 |    |    |-- days_of_drip_extension: long (nullable = true)
 |    |    |-- days_of_premium: long (nullable = true)
 |    |    |-- description: string (nullable = true)
 |    |    |-- end_date: long (nullable = true)
 |    |    |-- giver_coin_reward: long (nullable = true)
 |    |    |-- icon_format: string (nullable = true)
 |    |    |-- icon_height: long (nullable = true)
 |    |    |-- icon_url: string (nullable = true)
 |    |    |-- icon_width: long (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- is_enabled: boolean (nullable = true)
 |    |    |-- is_new: boolean (nullable = true)
 |    |    |-- name: str

In [7]:
subs_test.show(3, truncate=False)
comments_test.show(3, truncate=False)

+-------------+-------------------+--------+-------------------+--------------+------------------+-----------------------------+----------------------+---------------------+------------------------+-----------------+-----------------------+-----------------+---------------+--------------------+--------------+--------+--------+------------+--------+------------------+------------+-----------+---------------+-------------+---------------------+------+------+------------------+------+------+----------------+-------+-------------------+----------------------+------------------+-------+--------+---------------------------+--------------------+-------------------+------------------------------------+---------------+---------------------+---------------+------+-----+------------------------+--------------+----------+---------+------------+--------------+-------+-----------------------+---------------------------------------------------------------------------------+------+---------+-------+--

# For Submissions

In [7]:
from pyspark.sql.types import *

submission_schema = StructType([
    StructField("id", StringType(), True),
    StructField("author", StringType(), True),
    StructField("created_utc", LongType(), True),
    StructField("title", StringType(), True),
    StructField("selftext", StringType(), True),
    StructField("score", LongType(), True),
    StructField("num_comments", LongType(), True),
    StructField("subreddit", StringType(), True),
    StructField("is_self", BooleanType(), True),
    StructField("link_flair_text", StringType(), True),
    StructField("url", StringType(), True),
    StructField("removed_by_category", StringType(), True)
])

In [8]:
submissions_raw = spark.read.schema(submission_schema).json(RAW_SUBMISSIONS_PATH)

In [9]:
print("Submissions count:", submissions_raw.count())
submissions_raw.printSchema()
submissions_raw.show(5, truncate=False)

Submissions count: 1955840
root
 |-- id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- title: string (nullable = true)
 |-- selftext: string (nullable = true)
 |-- score: long (nullable = true)
 |-- num_comments: long (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- is_self: boolean (nullable = true)
 |-- link_flair_text: string (nullable = true)
 |-- url: string (nullable = true)
 |-- removed_by_category: string (nullable = true)

+------+-------------+-----------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+-----+------------+--------------+-------+---------------+------------------------------------------------------------------------------------------------------+-------------------+
|id    |author       |created_utc|title                       

In [11]:
submissions_raw.selectExpr(
    "min(created_utc) as min_time",
    "max(created_utc) as max_time"
).show()

[Stage 15:================================================>       (44 + 7) / 51]

+----------+----------+
|  min_time|  max_time|
+----------+----------+
|1577836853|1672530717|
+----------+----------+



In [12]:
submissions_raw.select("author").distinct().count()

668271

In [10]:
# after: submissions_raw.count(), printSchema(), show()

SUBMISSIONS_RAW_PARQUET_PATH = "/project/macss/amritap1/redditproject/data/processed/submissions_raw_parquet"

submissions_raw.write.mode("overwrite").parquet(SUBMISSIONS_RAW_PARQUET_PATH)

print("Submissions written successfully")

26/04/19 19:12:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 19:12:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 19:12:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 19:12:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 19:12:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 19:12:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 19:12:53 WARN MemoryManager: Total allocation exceeds 95.00% 

Submissions written successfully


### Data coverage validation - Submissions

To verify temporal completeness, I computed the minimum and maximum values of `created_utc` in the submissions dataset.

**Result:**
- Minimum timestamp: 1577836853 (~ Jan 1, 2020)
- Maximum timestamp: 1672530717 (~ Dec 31, 2022)

This validation ensures that downstream temporal and behavioral analyses are based on complete coverage of the target timeframe.

# For Comments

In [13]:
from pyspark.sql.types import *

comment_schema = StructType([
    StructField("id", StringType(), True),
    StructField("author", StringType(), True),
    StructField("created_utc", LongType(), True),
    StructField("body", StringType(), True),
    StructField("score", LongType(), True),
    StructField("subreddit", StringType(), True),
    StructField("author_flair_text", StringType(), True),
    StructField("controversiality", LongType(), True),
    StructField("is_submitter", BooleanType(), True),
    StructField("removed_by_category", StringType(), True),
    StructField("link_id", StringType(), True),
    StructField("parent_id", StringType(), True)
])

In [14]:
comments_test = spark.read.schema(comment_schema).json(
    "/project/macss/amritap1/redditproject/data/comments/RC_2020-01_subreddit.json"
)

print("Comments test count:", comments_test.count())
comments_test.printSchema()
comments_test.show(5, truncate=False)

Comments test count: 512967
root
 |-- id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- body: string (nullable = true)
 |-- score: long (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- controversiality: long (nullable = true)
 |-- is_submitter: boolean (nullable = true)
 |-- removed_by_category: string (nullable = true)
 |-- link_id: string (nullable = true)
 |-- parent_id: string (nullable = true)

+-------+-------------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [15]:
comments_test.selectExpr(
    "min(created_utc) as min_time",
    "max(created_utc) as max_time"
).show()

[Stage 28:====================================>                     (5 + 3) / 8]

+----------+----------+
|  min_time|  max_time|
+----------+----------+
|1577836804|1580515198|
+----------+----------+



In [16]:
RAW_COMMENTS_PARQUET_PATH = "/project/macss/amritap1/redditproject/data/processed/comments_raw_parquet"

In [20]:
# from pyspark.sql.functions import col, from_unixtime, to_timestamp, to_date, trim
import os
import glob

In [21]:
comment_files = sorted(glob.glob("/project/macss/amritap1/redditproject/data/comments/*.json"))

print("Number of comment files:", len(comment_files))
print("First 5 files:")
for f in comment_files[:5]:
    print(f)

Number of comment files: 36
First 5 files:
/project/macss/amritap1/redditproject/data/comments/RC_2020-01_subreddit.json
/project/macss/amritap1/redditproject/data/comments/RC_2020-02_subreddit.json
/project/macss/amritap1/redditproject/data/comments/RC_2020-03_subreddit.json
/project/macss/amritap1/redditproject/data/comments/RC_2020-04_subreddit.json
/project/macss/amritap1/redditproject/data/comments/RC_2020-05_subreddit.json


In [22]:
import shutil
import os

if os.path.exists(RAW_COMMENTS_PARQUET_PATH):
    shutil.rmtree(RAW_COMMENTS_PARQUET_PATH)
    print("Old raw comments parquet folder removed")
else:
    print("No existing raw comments parquet folder found")

Old raw comments parquet folder removed


In [23]:
small_batch = comment_files[:3]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2020-01_subreddit.json


26/04/19 18:11:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:11:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2020-01_subreddit.json

Processing file 2/3: RC_2020-02_subreddit.json


26/04/19 18:12:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2020-02_subreddit.json

Processing file 3/3: RC_2020-03_subreddit.json


26/04/19 18:12:04 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:12:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:12:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:12:05 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:12:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:12:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
[Stage 62:=====================================================>  (25 

Finished: RC_2020-03_subreddit.json


In [24]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows:", comments_check.count())
comments_check.printSchema()
comments_check.show(5, truncate=False)

Written rows: 4162731
root
 |-- id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- body: string (nullable = true)
 |-- score: long (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- controversiality: long (nullable = true)
 |-- is_submitter: boolean (nullable = true)
 |-- removed_by_category: string (nullable = true)
 |-- link_id: string (nullable = true)
 |-- parent_id: string (nullable = true)

+-------+---------------+-----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author         |created_utc|

In [25]:
small_batch = comment_files[3:6]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2020-04_subreddit.json


26/04/19 18:22:21 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:22:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:22:23 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

Finished: RC_2020-04_subreddit.json

Processing file 2/3: RC_2020-05_subreddit.json


26/04/19 18:22:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:22:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:22:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:22:31 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2020-05_subreddit.json

Processing file 3/3: RC_2020-06_subreddit.json


26/04/19 18:22:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:22:35 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
[Stage 70:==================================================>     (17 + 2) / 19]

Finished: RC_2020-06_subreddit.json


In [26]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows so far:", comments_check.count())
comments_check.show(5, truncate=False)

Written rows so far: 9445830
+-------+-------------------+-----------+---------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author             |created_utc|body                                   |score|subreddit     |author_flair_text|controversiality|is_submitter|removed_by_category|link_id  |parent_id |
+-------+-------------------+-----------+---------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|fmwt08c|BigBoiBenis        |1586452504 |Wow that is almost like mine\n5k-31k-7k|3    |wallstreetbets|NULL             |0               |false       |NULL               |t3_fxqidb|t1_fmwsx4u|
|fmwt09i|[deleted]          |1586452505 |[deleted]                              |1    |wallstreetbets|NULL             |0               |false       |NULL               |t3_fxqidb|t1_fmwsx4u|
|fmwt0av|Au

In [27]:
small_batch = comment_files[6:9]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2020-07_subreddit.json


26/04/19 18:23:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:23:44 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:23:44 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:23:44 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:23:44 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

Finished: RC_2020-07_subreddit.json

Processing file 2/3: RC_2020-08_subreddit.json


26/04/19 18:23:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:23:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:23:48 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:23:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2020-08_subreddit.json

Processing file 3/3: RC_2020-09_subreddit.json


26/04/19 18:23:52 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
[Stage 78:==================================>                      (9 + 6) / 15]

Finished: RC_2020-09_subreddit.json


In [28]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows so far:", comments_check.count())
comments_check.show(5, truncate=False)

Written rows so far: 13716877
+-------+-------------------+-----------+---------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author             |created_utc|body                                   |score|subreddit     |author_flair_text|controversiality|is_submitter|removed_by_category|link_id  |parent_id |
+-------+-------------------+-----------+---------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|fmwt08c|BigBoiBenis        |1586452504 |Wow that is almost like mine\n5k-31k-7k|3    |wallstreetbets|NULL             |0               |false       |NULL               |t3_fxqidb|t1_fmwsx4u|
|fmwt09i|[deleted]          |1586452505 |[deleted]                              |1    |wallstreetbets|NULL             |0               |false       |NULL               |t3_fxqidb|t1_fmwsx4u|
|fmwt0av|A

In [29]:
small_batch = comment_files[9:12]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2020-10_subreddit.json


26/04/19 18:25:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:25:41 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:25:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2020-10_subreddit.json

Processing file 2/3: RC_2020-11_subreddit.json


26/04/19 18:25:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:25:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:25:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2020-11_subreddit.json

Processing file 3/3: RC_2020-12_subreddit.json


26/04/19 18:25:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:25:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:25:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:25:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
[Stage 86:==================================================>     (17 + 2) / 19]

Finished: RC_2020-12_subreddit.json


In [30]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows so far:", comments_check.count())
comments_check.show(5, truncate=False)

Written rows so far: 18194372
+-------+-------------------+-----------+---------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author             |created_utc|body                                   |score|subreddit     |author_flair_text|controversiality|is_submitter|removed_by_category|link_id  |parent_id |
+-------+-------------------+-----------+---------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|fmwt08c|BigBoiBenis        |1586452504 |Wow that is almost like mine\n5k-31k-7k|3    |wallstreetbets|NULL             |0               |false       |NULL               |t3_fxqidb|t1_fmwsx4u|
|fmwt09i|[deleted]          |1586452505 |[deleted]                              |1    |wallstreetbets|NULL             |0               |false       |NULL               |t3_fxqidb|t1_fmwsx4u|
|fmwt0av|A

In [31]:
small_batch = comment_files[12:15]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2021-01_subreddit.json


26/04/19 18:26:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:26:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:26:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:26:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:26:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:26:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:26:28 WARN MemoryManager: Total allocation exceeds 95.00% 

Finished: RC_2021-01_subreddit.json

Processing file 2/3: RC_2021-02_subreddit.json


26/04/19 18:26:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:26:57 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:26:59 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:26:59 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:26:59 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:27:00 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:27:01 WARN MemoryManager: Total allocation exceeds 95.00% 

Finished: RC_2021-02_subreddit.json

Processing file 3/3: RC_2021-03_subreddit.json


26/04/19 18:27:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:27:14 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:27:14 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:27:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:27:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:27:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:27:18 WARN MemoryManager: Total allocation exceeds 95.00% 

Finished: RC_2021-03_subreddit.json


In [32]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows so far:", comments_check.count())
comments_check.show(5, truncate=False)

Written rows so far: 37223167
+-------+-------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author       |created_utc|body                                                                                                                                                                                                                                                                                                                                                       

In [33]:
small_batch = comment_files[15:18]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2021-04_subreddit.json


26/04/19 18:28:41 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:28:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:28:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:28:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:28:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2021-04_subreddit.json

Processing file 2/3: RC_2021-05_subreddit.json


26/04/19 18:28:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:28:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:28:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

Finished: RC_2021-05_subreddit.json

Processing file 3/3: RC_2021-06_subreddit.json


26/04/19 18:28:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:28:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:28:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:28:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:28:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:28:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
[Stage 102:==================================================>    (21 

Finished: RC_2021-06_subreddit.json


In [34]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows so far:", comments_check.count())
comments_check.show(5, truncate=False)

Written rows so far: 42458288
+-------+-------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author       |created_utc|body                                                                                                                                                                                                                                                                                                                                                       

In [35]:
small_batch = comment_files[18:21]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2021-07_subreddit.json


26/04/19 18:29:38 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

Finished: RC_2021-07_subreddit.json

Processing file 2/3: RC_2021-08_subreddit.json


26/04/19 18:29:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:29:44 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

Finished: RC_2021-08_subreddit.json

Processing file 3/3: RC_2021-09_subreddit.json


26/04/19 18:29:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
[Stage 110:==================================================>    (10 + 1) / 11]

Finished: RC_2021-09_subreddit.json


In [36]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows so far:", comments_check.count())
comments_check.show(5, truncate=False)

Written rows so far: 45575594
+-------+-------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author       |created_utc|body                                                                                                                                                                                                                                                                                                                                                       

In [37]:
small_batch = comment_files[21:24]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2021-10_subreddit.json


26/04/19 18:31:29 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:31:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2021-10_subreddit.json

Processing file 2/3: RC_2021-11_subreddit.json


26/04/19 18:31:47 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2021-11_subreddit.json

Processing file 3/3: RC_2021-12_subreddit.json


[Stage 118:========================================>               (8 + 3) / 11]

Finished: RC_2021-12_subreddit.json


In [38]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows so far:", comments_check.count())
comments_check.show(5, truncate=False)

Written rows so far: 48555551
+-------+-------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author       |created_utc|body                                                                                                                                                                                                                                                                                                                                                       

In [40]:
small_batch = comment_files[24:27]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2022-01_subreddit.json


26/04/19 18:32:54 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:32:58 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2022-01_subreddit.json

Processing file 2/3: RC_2022-02_subreddit.json


26/04/19 18:33:02 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2022-02_subreddit.json

Processing file 3/3: RC_2022-03_subreddit.json


26/04/19 18:33:03 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
[Stage 129:======================================>                 (9 + 4) / 13]

Finished: RC_2022-03_subreddit.json


In [41]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows so far:", comments_check.count())
comments_check.show(5, truncate=False)

Written rows so far: 55862591
+-------+-------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author       |created_utc|body                                                                                                                                                                                                                                                                                                                                                       

In [42]:
small_batch = comment_files[27:30]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2022-04_subreddit.json


26/04/19 18:33:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:33:39 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:33:41 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2022-04_subreddit.json

Processing file 2/3: RC_2022-05_subreddit.json


26/04/19 18:33:43 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:33:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2022-05_subreddit.json

Processing file 3/3: RC_2022-06_subreddit.json


[Stage 137:======================================>                 (9 + 4) / 13]

Finished: RC_2022-06_subreddit.json


In [43]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows so far:", comments_check.count())
comments_check.show(5, truncate=False)

Written rows so far: 59354457
+-------+-------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author       |created_utc|body                                                                                                                                                                                                                                                                                                                                                       

In [44]:
small_batch = comment_files[30:33]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2022-07_subreddit.json


26/04/19 18:34:16 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:34:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:34:18 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:34:20 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2022-07_subreddit.json

Processing file 2/3: RC_2022-08_subreddit.json


26/04/19 18:34:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:34:22 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:34:26 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2022-08_subreddit.json

Processing file 3/3: RC_2022-09_subreddit.json


26/04/19 18:34:27 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:34:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
[Stage 145:====================================>                   (9 + 5) / 14]

Finished: RC_2022-09_subreddit.json


In [45]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows so far:", comments_check.count())
comments_check.show(5, truncate=False)

Written rows so far: 63358211
+-------+-------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author       |created_utc|body                                                                                                                                                                                                                                                                                                                                                       

In [46]:
small_batch = comment_files[33:36]

for i, file_path in enumerate(small_batch):
    print(f"\nProcessing file {i+1}/{len(small_batch)}: {os.path.basename(file_path)}")

    comments_month = spark.read.schema(comment_schema).json(file_path)

    comments_month.write.mode("append").parquet(RAW_COMMENTS_PARQUET_PATH)

    print(f"Finished: {os.path.basename(file_path)}")


Processing file 1/3: RC_2022-10_subreddit.json


26/04/19 18:35:10 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:35:12 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

Finished: RC_2022-10_subreddit.json

Processing file 2/3: RC_2022-11_subreddit.json


26/04/19 18:35:13 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/19 18:35:17 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Finished: RC_2022-11_subreddit.json

Processing file 3/3: RC_2022-12_subreddit.json


26/04/19 18:35:19 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
[Stage 153:=============================================>          (9 + 2) / 11]

Finished: RC_2022-12_subreddit.json


In [47]:
comments_check = spark.read.parquet(RAW_COMMENTS_PARQUET_PATH)

print("Written rows so far:", comments_check.count())
comments_check.show(5, truncate=False)

Written rows so far: 66444342
+-------+-------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author       |created_utc|body                                                                                                                                                                                                                                                                                                                                                       

In [11]:
old_comments = spark.read.parquet("/project/macss/amritap1/redditproject/data/processed/comments_parquet")
new_comments = spark.read.parquet("/project/macss/amritap1/redditproject/data/processed/comments_raw_parquet")

In [12]:
print("Old comments_parquet rows:", old_comments.count())
print("New comments_raw_parquet rows:", new_comments.count())

Old comments_parquet rows: 26317557


[Stage 11:======================================================> (56 + 2) / 58]

New comments_raw_parquet rows: 66444342


In [13]:
print("OLD schema")
old_comments.printSchema()

print("NEW schema")
new_comments.printSchema()

OLD schema
root
 |-- id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- body: string (nullable = true)
 |-- score: long (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- controversiality: long (nullable = true)
 |-- is_submitter: boolean (nullable = true)
 |-- removed_by_category: string (nullable = true)
 |-- link_id: string (nullable = true)
 |-- parent_id: string (nullable = true)

NEW schema
root
 |-- id: string (nullable = true)
 |-- author: string (nullable = true)
 |-- created_utc: long (nullable = true)
 |-- body: string (nullable = true)
 |-- score: long (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- author_flair_text: string (nullable = true)
 |-- controversiality: long (nullable = true)
 |-- is_submitter: boolean (nullable = true)
 |-- removed_by_category: string (nullable = true)
 |-- link_id: string (nullable = true)
 |-- parent_id: 

In [14]:
old_comments.show(5, truncate=False)
new_comments.show(5, truncate=False)

+-------+-------------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+--------------+-----------------+----------------+------------+-------------------+---------+----------+
|id     |author       |created_utc|body                                                                                                                                                                                                                                                                                                                                                                                     

In [15]:
old_comments.selectExpr(
    "min(created_utc) as min_time",
    "max(created_utc) as max_time"
).show()

new_comments.selectExpr(
    "min(created_utc) as min_time",
    "max(created_utc) as max_time"
).show()

+----------+----------+
|  min_time|  max_time|
+----------+----------+
|1577836804|1612137599|
+----------+----------+



[Stage 19:======================================================> (56 + 2) / 58]

+----------+----------+
|  min_time|  max_time|
+----------+----------+
|1577836804|1672531193|
+----------+----------+



In [16]:
print("Old distinct ids:", old_comments.select("id").distinct().count())
print("New distinct ids:", new_comments.select("id").distinct().count())

26/04/19 19:19:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:19:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:19:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:19:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:19:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:19:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:19:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:19:46 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:19:58 WARN RowBasedKeyValueBatch: Calling spill() on

Old distinct ids: 26317557


26/04/19 19:20:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:20:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:20:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:20:06 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:20:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:20:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:20:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:20:07 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/04/19 19:20:08 WARN RowBasedKeyValueBatch: Calling spill() on

New distinct ids: 62790822


In [17]:
import shutil

shutil.rmtree("/project/macss/amritap1/redditproject/data/processed/comments_parquet")

print("Old comments_parquet removed")

Old comments_parquet removed


### Large-scale Reddit comment ingestion (r/wallstreetbets, 2020–2022)

The full comment dataset was ingested from raw monthly JSON files into a structured Parquet format using PySpark.

#### Approach

Initial attempts to load all comment files simultaneously resulted in memory failures due to:
- large file sizes (multi-GB monthly files)
- nested JSON structure
- Spark schema inference overhead

To ensure stability, a **batched ingestion strategy** was implemented.

#### Batched processing strategy

- Files were processed in small batches (3 months at a time)
- Each batch:
  - loads JSON using an explicit schema
  - writes directly to Parquet (`append` mode)
- No cleaning or transformation was applied at this stage

This design:
- avoids JVM heap crashes
- reduces memory pressure
- creates a reusable intermediate dataset

#### Sequential processing across years

Files were processed in chronological order using filename sorting.

Because filenames follow the structure `RC_YYYY-MM`, batching naturally spans across year boundaries without requiring special handling.

This keeps the pipeline simple while preserving temporal ordering for downstream analysis.

#### Result

- Full dataset successfully ingested
- Total comments processed: ~66 million+
- Output stored as partitioned Parquet files

#### Observations from raw data

The raw dataset includes:
- `[deleted]` and `[removed]` content
- AutoModerator and bot-generated posts
- highly variable text length and quality
- nested Reddit metadata fields

These artifacts were intentionally preserved at this stage to maintain data fidelity.

#### Next step

A separate preprocessing stage will:
- clean text (remove deleted/empty content)
- normalize timestamps
- filter low-signal observations
- prepare data for NLP and behavioral analysis

This separation ensures that ingestion remains robust, while transformation remains flexible and reproducible.